# SpineLIT — MRI Segmentation (nnU-Net) + Detection (YOLO) Training
Runs the SPIDER -> nnU-Net dataset build, nnU-Net 3-D training, SPIDER -> YOLO dataset build, and YOLO detection training on a Colab T4 GPU.

**Before starting:** Runtime > Change runtime type > **GPU (T4)**. Keep `colab_training.ipynb` for the severity (ConvNeXt Pfirrmann) model — this notebook trains the *upstream* anatomy models this time.

## 0. GPU check + repo + Drive

In [ ]:
import torch
if not torch.cuda.is_available():
    raise SystemExit('No GPU detected. Runtime > Change runtime type > GPU (T4), restart, re-run.')
print('GPU:', torch.cuda.get_device_name(0), '| CUDA', torch.version.cuda)


In [ ]:
# 0b. Mount Google Drive (where SPIDER_data lives)
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
print('MyDrive root:', sorted(p.name for p in Path('/content/drive/MyDrive').iterdir())[:12])
assert Path('/content/drive/MyDrive/SPIDER_data').is_dir(), \
    'SPIDER_data not found in MyDrive - upload it or set SPIDER in the next cell to the real path'


In [ ]:
%cd /content
import os
GITHUB_URL = 'https://github.com/codermisba/spinelit-ai.git'
if not os.path.exists('/content/spine-foundation'):
    !git clone {GITHUB_URL} spine-foundation
%cd /content/spine-foundation/
!git fetch origin && !git reset --hard origin/main
!git log --oneline -1


In [ ]:
# Install ONLY what this stage needs. Do NOT pip install torch (Colab's build is CUDA).
!pip install -q nnunetv2 ultralytics SimpleITK
import SimpleITK; print('SimpleITK', SimpleITK.__version__)
from ultralytics import YOLO  # verify import
print('ultralytics OK')


## Stage 1 — nnU-Net segmentation of vertebrae / discs / spinal canal
Masks are the SPIDER **expert** segmentations; semantic labels = 1 vertebra, 2 disc, 3 spinal canal.

In [ ]:
# 1a. Copy SPIDER_data to the local Colab disk FIRST.
# SimpleITK cannot reliably read .mha through the Drive FUSE mount.
import shutil, time
from pathlib import Path
SRC = Path('/content/drive/MyDrive/SPIDER_data')
DST = Path('/content/SPIDER_data')
want_img = len(list((SRC / 'images').glob('*.mha'))) if (SRC / 'images').is_dir() else 0
want_msk = len(list((SRC / 'masks').glob('*.mha'))) if (SRC / 'masks').is_dir() else 0
print('Drive has images/masks:', want_img, want_msk)
assert want_img and want_msk, 'SPIDER_data/images or /masks missing on Drive'
for attempt in range(3):
    try:
        if not (DST / 'images').exists(): shutil.copytree(SRC / 'images', DST / 'images')
        if not (DST / 'masks').exists(): shutil.copytree(SRC / 'masks', DST / 'masks')
        break
    except Exception as exc:
        print('copy attempt', attempt + 1, 'failed:', str(exc)[:70])
        time.sleep(5)
got_img = len(list((DST / 'images').glob('*.mha')))
got_msk = len(list((DST / 'masks').glob('*.mha')))
print('Copied locally: images/masks =', got_img, got_msk)
assert got_img == want_img and got_msk == want_msk, 'local copy incomplete - rerun cell'
print('LOCAL COPY READY')
# The SPIDER metadata CSVs carry the Pfirrmann ground truth - fetch them if missing.
import requests
for csv_name in ('overview.csv', 'radiological_gradings.csv'):
    dst = DST / csv_name
    if dst.exists() and dst.stat().st_size > 0:
        print('have', csv_name)
        continue
    url = 'https://zenodo.org/records/10159290/files/' + csv_name
    print('downloading', csv_name, '...')
    r = requests.get(url, stream=True, timeout=180)
    r.raise_for_status()
    with open(dst, 'wb') as fh:
        for chunk in r.iter_content(1 << 16):
            fh.write(chunk)
    print('  saved', dst.stat().st_size, 'bytes')


In [ ]:
# 1b. Build the nnU-Net raw dataset from LOCAL copies of the .mha volumes
!python nnunet_prep.py --images_dir /content/SPIDER_data/images \
    --masks_dir /content/SPIDER_data/masks --out_base /content/nnUNet_raw 2>&1 | tail -n 10
import pathlib
ds = pathlib.Path('/content/nnUNet_raw/Dataset501_SPIDERMRI')
print('dataset.json:', (ds / 'dataset.json').is_file(),
      '| imagesTr:', len(list((ds / 'imagesTr').glob('*.nii.gz'))),
      '| labelsTr:', len(list((ds / 'labelsTr').glob('*.nii.gz'))))
assert (ds / 'dataset.json').is_file(), 'dataset.json missing - see errors above'


In [ ]:
# 2. Plan/preprocess (nnU-Net auto-configures the U-Net for our data)
import os
os.environ['nnUNet_raw'] = '/content/nnUNet_raw'
os.environ['nnUNet_preprocessed'] = '/content/nnUNet_preprocessed'
os.environ['nnUNet_results'] = '/content/nnUNet_results'
os.makedirs('/content/nnUNet_preprocessed', exist_ok=True)
os.makedirs('/content/nnUNet_results', exist_ok=True)
import pathlib
assert pathlib.Path('/content/nnUNet_raw/Dataset501_SPIDERMRI/dataset.json').is_file(),     'dataset.json missing - run the Stage 1 build cell first'
!nnUNetv2_plan_and_preprocess -d 501 --verify_dataset_integrity 2>&1 | tail -n 5


**Training:** 3-D segmentation on a T4 takes hours (~30-60 min/fold). Run the full 5-fold
cross-validation if you have runtime budget; for the demo you can train a single fold. Checkpoint
appears under `/content/nnUNet_results/Dataset501_SPIDERMRI/nnUNetTrainer__nnUNetPlans__3d_fullres/fold_all`.

In [ ]:
# 3. Train (use fold_all for the demo; 'all' trains full-data as a single fold)
os.environ['nnUNet_raw'] = '/content/nnUNet_raw'
os.environ['nnUNet_preprocessed'] = '/content/nnUNet_preprocessed'
os.environ['nnUNet_results'] = '/content/nnUNet_results'
!nnUNetv2_train 501 3d_fullres all 2>&1 | tail -n 8


In [ ]:
# 4. Quick inference sanity check on one held-out/sample volume
os.environ.update(nnUNet_raw='/content/nnUNet_raw',
                  nnUNet_preprocessed='/content/nnUNet_preprocessed',
                  nnUNet_results='/content/nnUNet_results')
import glob, SimpleITK as sitk, numpy as np
sample = glob.glob('/content/nnUNet_raw/Dataset501_SPIDERMRI/imagesTs/*.nii.gz')
if not sample:
    sample = glob.glob('/content/nnUNet_raw/Dataset501_SPIDERMRI/imagesTr/*.nii.gz')
print('sample:', sample[:1])
!mkdir -p /content/nnUNet_pred
!nnUNetv2_predict -i /content/nnUNet_pred_in -o /content/nnUNet_pred_out -d 501 \
    -c 3d_fullres -f all 2>/dev/null || echo "(run predict after training completes; create imagesTs folder with the images you want segmented)" 


## Stage 2 — YOLO detection of vertebrae + discs (mid-sagittal frame)
Boxes are derived automatically from the SPIDER masks — zero manual labelling.

In [ ]:
# 1c. Build YOLO detection dataset from LOCAL copies of the .mha volumes
!python yolo_prep.py --images_dir /content/SPIDER_data/images \
    --masks_dir /content/SPIDER_data/masks --out_dir dataset/yolo 2>&1 | tail -n 8


In [ ]:
# 6. Create the Ultralytics dataset config
from pathlib import Path
base = Path('dataset/yolo').resolve()
yaml = f"""path: {base}
train: train.txt
val: val.txt
names:
  0: vertebra
  1: disc
"""
Path('dataset/yolo/spider.yaml').write_text(yaml)
print(Path('dataset/yolo/spider.yaml').read_text())


**Training:** about 30-60 minutes on a T4 at imgsz=640. `best.pt` is the demo checkpoint.

In [ ]:
# 7. Train YOLO detection
from ultralytics import YOLO
model = YOLO('yolo11n.pt')          # smallest COCO-pretrained YOLOv11
model.train(data='dataset/yolo/spider.yaml', epochs=80, imgsz=640,
            device=0, patience=15, project='dataset/yolo/runs')
metrics = model.val()
print('mAP50-95:', round(float(metrics.box.map), 3))


In [ ]:
# 8. Save both trained models to Drive (reuse for inference / API)
DEST = '/content/drive/MyDrive/spine-modules'
!mkdir -p {DEST}
!zip -qr {DEST}/nnunet_spider.zip /content/nnUNet_results/Dataset501_SPIDERMRI 2>/dev/null
from pathlib import Path
best = list(Path('dataset/yolo/runs').rglob('best.pt'))
if best:
    import shutil; shutil.copy(str(best[0]), f'{DEST}/spider_yolov11.pt')
    print('saved ->', f'{DEST}/spider_yolov11.pt', best[0])
from google.colab import files
files.download(f'{DEST}/spider_yolov11.pt') if best else None


**Next steps back in SpineLIT:** YOLO supplies disc/vertebra boxes, nnU-Net supplies 3-D masks
(centroids + disc-space geometry), the ConvNeXt severity model grades Pfirrmann, and `explainability.py`
produces Grad-CAM — wired together in the vision module of the API.